### 1. CLEANING FOR SBERT

1.1 Data Loading


In [16]:
import pandas as pd
df= pd.read_csv('/Users/prabinrimal/Desktop/New>>>>>/text extraction/final_dataset.csv')
df

,resume_id,resume_text,category
0,0,Hieu-Thi Luong\ncontact@hieuthi.com | https://...,AI ML
1,1,bitbucket.org/mikkujacob | github.com/mikhailj...,AI ML
2,2,"MERT KIRAY\nGörresstr. 3, 80798 München\n+4915...",AI ML
3,3,Adam Pardyl Email : adam@pardyl.com\nPhD Candi...,AI ML
4,4,Karystinaios\nEmmanouil AI Researcher — Music ...,AI ML
...,...,...,...
1424,1530,"Aidan M. Levinson\n5271 Wheatland Drive, Zions...",Technology Consultant
1425,1531,Arturo Hernandez Jr.\nEducation\nCORNELL COLLE...,Technology Consultant
1426,1532,Jide Akanni\njide.akanni361@gmail.com\nCell (2...,Technology Consultant
1427,1533,Kurt Mansperger 845-797-0506  @kurtmansperger...,Technology Consultant


1.2 Noise Removal

In [17]:
df["resume_text"].head()

0    Hieu-Thi Luong\ncontact@hieuthi.com | https://...
1    bitbucket.org/mikkujacob | github.com/mikhailj...
2    MERT KIRAY\nGörresstr. 3, 80798 München\n+4915...
3    Adam Pardyl Email : adam@pardyl.com\nPhD Candi...
4    Karystinaios\nEmmanouil AI Researcher — Music ...
Name: resume_text, dtype: str

In [18]:
import re
import unicodedata
import spacy
from tqdm.auto import tqdm

NOISE_PHRASES = [
    r"curriculum vitae", r"\bresume\b", r"personal name", r"personal information",
    r"references available upon request", r"linkedin", r"github", r"portfolio",
    r"novypro", r"tableau public",
]

HEADER_WORDS = 60

SECTION_START_PATTERN = re.compile(
    r"(?i)\b(resume\s+objective|objective|professional\s+summary|summary|profile|"
    r"experience|work\s+experience|employment\s+history|education|skills|"
    r"technical\s+skills|languages)\b"
)

#unicode and stylized text normalization
def basic_clean(text):
    cleaned_chars = []
    for ch in str(text):
        if ch in "\n\r\t":
            cleaned_chars.append(ch)
        elif unicodedata.category(ch).startswith("C"):
            continue
        elif ch.isprintable():
            cleaned_chars.append(ch)
    text = "".join(cleaned_chars)
    text = re.sub(r"\(cid:\d+\)", "", text)
    text = re.sub(r"(?:[A-Z]\s){3,}[A-Z]", lambda m: m.group(0).replace(" ", ""), text)
    return text


def remove_contact_and_pii(text):
    email_pattern = re.compile(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b")
    phone_pattern = re.compile(r"(?<!\w)\(?\+?\d[\d\s().-]{6,}\d\)?(?!\w)")

    street_address_pattern = re.compile(
        r"\b\d{1,5}\s+(?:[A-Za-z0-9.\-\' ]{1,8}\s+){0,6}(?:Street|St|Road|Rd|Avenue|Ave|"
        r"Boulevard|Blvd|Lane|Ln|Drive|Dr|Way|Court|Ct|Circle|Cir|Place|Pl|Terrace|Ter|"
        r"Parkway|Pkwy|Trail|Trl|Highway|Hwy|Rodovia)\b[^,.;\n]{0,60}", re.IGNORECASE)

    city_state_zip_pattern = re.compile(
        r"\b[A-Za-z]+(?:\s+[A-Za-z]+){0,3}\s*,\s*[A-Za-z]{2}\s+\d{5}(?:-\d{4})?\b")
    zip_city_state_pattern = re.compile(
        r"\b\d{5}(?:-\d{4})?\s+[A-Za-z]+(?:\s+[A-Za-z]+){0,3}\s*,\s*[A-Za-z]{2}\b")

    labeled_field_pattern = re.compile(
        r"(?i)\b(address|country of residence|nationality|date of birth|dob|"
        r"marital status)\s*:?\s*.{0,120}?(?=\b(?:Phone|Email|Address|Country|Nationality|"
        r"Objective|Summary|Experience|Education|Skills)\b|[.\n]|$)")

    text = email_pattern.sub(" ", text)
    text = phone_pattern.sub(" ", text)
    text = street_address_pattern.sub(" ", text)
    text = city_state_zip_pattern.sub(" ", text)
    text = zip_city_state_pattern.sub(" ", text)
    text = labeled_field_pattern.sub(" ", text)

    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(
        r"\b(?:[a-z0-9-]+\.)+(?:com|org|net|edu|gov|io|co|uk|in|biz|info)(?:/[^\s]*)?",
        " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\b[\w.+-]+@(?=\s|$)", " ", text)
    return text


def remove_boilerplate(text):
    section_match = SECTION_START_PATTERN.search(text[:1200])
    if section_match and section_match.start() > 0:
        text = text[section_match.start():]

    for phrase in NOISE_PHRASES:
        text = re.sub(phrase, " ", text, flags=re.IGNORECASE)
    orphan_label_pattern = re.compile(
        r"(?i)\b(phone(?:\s*number)?|mobile|cell|tel(?:ephone)?|e[-\s]?mail)"
        r"\s*[:\-]?\s*(?=[,()\[\]\s]|$)")
    return orphan_label_pattern.sub(" ", text)


def final_normalize(text):
    text = re.sub(r"[•▪◦‣➤→✓❖✦|]", " ", text)
    text = re.sub(r"[^\w\s.,!?;:\-]", " ", text)
    text = re.sub(r"[-]+[–]+[—]+[.]+[_]+", " ", text)
    text = re.sub(r"\(\s*\)", " ", text)
    text = re.sub(r"\[\s*\]", " ", text)
    text = re.sub(r"\s*,\s*,+", ",", text)
    text = re.sub(r"(^|\s),", r"\1", text)
    text = re.sub(r"[.,;:]{2,}", ".", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def split_header_body(text, header_words=HEADER_WORDS):
    words = text.split()
    return " ".join(words[:header_words]), " ".join(words[header_words:])


def remove_names_and_places(text, nlp):
    doc = nlp(text)
    return remove_names_and_places_from_doc(doc)


def remove_names_and_places_from_doc(doc):
    text = doc.text
    spans = [(e.start_char, e.end_char) for e in doc.ents if e.label_ in {"PERSON", "GPE", "LOC"}]
    for start, end in sorted(spans, key=lambda s: -s[0]):
        text = text[:start] + " " + text[end:]
    return text


# Pipeline
try:
    nlp = spacy.load("en_core_web_sm", enable=["ner"])
except OSError:
    nlp = None
    print("spaCy model 'en_core_web_sm' not installed; continuing without NER name/place removal.")

tqdm.pandas(desc="regex cleaning")
pre_ner = df["resume_text"].progress_apply(
    lambda t: remove_boilerplate(remove_contact_and_pii(basic_clean(t)))
)

headers, bodies = zip(*[split_header_body(t) for t in pre_ner])

if nlp is None:
    cleaned_headers = headers
else:
    cleaned_headers = [
        remove_names_and_places_from_doc(doc)
        for doc in tqdm(
            nlp.pipe(headers, batch_size=200),
            total=len(headers), desc="NER on headers"
        )
    ]

df["resume_text"] = [
    final_normalize(h + " " + b) for h, b in zip(cleaned_headers, bodies)
]

NER on headers: 100%|██████████| 1429/1429 [00:05<00:00, 268.99it/s]


In [19]:
df["resume_text"].head()

0    experience in speech processing, deepfake dete...
1    Education PhD Computer Science, Georgia Instit...
2    Education TUM, Master of Science Informatics, ...
3    Education Jagiellonian University PhD candidat...
4    Profile Postdoctoral AI researcher with expert...
Name: resume_text, dtype: str

In [20]:
pd.set_option("display.max_colwidth", 50)  # truncate long text with "..." like in the screenshot

df_original = pd.read_csv("/Users/prabinrimal/Desktop/New>>>>>/text extraction/final_dataset.csv")
df_original = df_original.dropna(subset=["resume_text"]).reset_index(drop=True)

comparison = pd.DataFrame({
    "resume_id": df_original["resume_id"].head(),
    "Resume_str": df_original["resume_text"].head().astype(str),
    "clean_text_resume": df["resume_text"].head().astype(str)
})

comparison

,resume_id,Resume_str,clean_text_resume
0,0,Hieu-Thi Luong\ncontact@hieuthi.com | https://...,"experience in speech processing, deepfake dete..."
1,1,bitbucket.org/mikkujacob | github.com/mikhailj...,"Education PhD Computer Science, Georgia Instit..."
2,2,"MERT KIRAY\nGörresstr. 3, 80798 München\n+4915...","Education TUM, Master of Science Informatics, ..."
3,3,Adam Pardyl Email : adam@pardyl.com\nPhD Candi...,Education Jagiellonian University PhD candidat...
4,4,Karystinaios\nEmmanouil AI Researcher — Music ...,Profile Postdoctoral AI researcher with expert...


In [21]:
df.to_csv("/Users/prabinrimal/Desktop/New>>>>>/ANOTHER/light_cleaned_SBERT.csv", index=False)

### 2. CLEANING FOR WORD2VEC


In [25]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

tqdm.pandas()

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng") 

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()
def _get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith("J"):
        return wordnet.ADJ
    elif treebank_tag.startswith("V"):
        return wordnet.VERB
    elif treebank_tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/prabinrimal/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/prabinrimal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/prabinrimal/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/prabinrimal/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/prabinrimal/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


2.1 Tokenization, Stopwords Removal and Lemmatization

In [26]:
def clean_word2vec(text):

    if pd.isna(text):
        return []

    text = str(text).lower()
    
    # Remove punctuation
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    words = word_tokenize(text)

    # Remove stopwords
    words = [w for w in words if w not in stop_words]

    # Lemmatize
    pos_tags = pos_tag(words)
    words = [lemmatizer.lemmatize(w, _get_wordnet_pos(tag)) for w, tag in pos_tags]

    # Remove short words
    words = [w for w in words if len(w) > 2]

    return words

In [ ]:
pd.set_option("display.max_colwidth", 50) 

new_df = df.copy()
new_df["resume_text"] = new_df["resume_text"].apply(clean_word2vec)

comparison_w2v = pd.DataFrame({
    "resume_id": df["resume_id"].head(),
    "Resume_str": df["resume_text"].head().astype(str),
    "clean_text_resume": new_df["resume_text"].head().astype(str)
})

comparison_w2v


,resume_id,Resume_str,clean_text_resume
0,0,"experience in speech processing, deepfake dete...","['experience', 'speech', 'process', 'deepfake'..."
1,1,"Education PhD Computer Science, Georgia Instit...","['education', 'phd', 'computer', 'science', 'g..."
2,2,"Education TUM, Master of Science Informatics, ...","['education', 'tum', 'master', 'science', 'inf..."
3,3,Education Jagiellonian University PhD candidat...,"['education', 'jagiellonian', 'university', 'p..."
4,4,Profile Postdoctoral AI researcher with expert...,"['profile', 'postdoctoral', 'researcher', 'exp..."


In [28]:
new_df.to_csv("/Users/prabinrimal/Desktop/New>>>>>/ANOTHER/cleaned_word2vec.csv", index=False)